# 05 - Validação

**Importação de bibliotecas**

In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline                               # organiza transformações em sequência
from sklearn.impute import SimpleImputer                            # preenche valores ausentes
from sklearn.preprocessing import StandardScaler, OneHotEncoder 
from sklearn.compose import ColumnTransformer
from sklearn.metrics import confusion_matrix                        # recebe os valores reais e as previsões do modelo e compara os dois

**Leitura do CSV**

In [2]:
df = pd.read_csv("data/heart_preparado.csv")

**Divisão de X e y**

In [3]:
y = df["has_disease"]

X = df.drop(columns=["has_disease", "id", "dataset"])

**Dividir treino e teste**

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

**Definindo as colunas**

In [5]:
numeric_cols = [
    "age",
    "trestbps",
    "chol",
    "thalch",
    "oldpeak",
    "ca"
]

categorical_cols = [
    "sex",
    "cp",
    "fbs",
    "restecg",
    "exang",
    "slope",
    "thal"
]

**Pipelines**

In [6]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())   
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_cols),
        ("cat", categorical_pipeline, categorical_cols)
    ]
)

**Unindo o pré-processamento com o modelo:**

In [7]:
model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000)) # limite maior de iterações para permitir que a regressão logística encontre a solução de otimização, reduzindo a possibilidade de não convergência com o limite padrão.
])

## Validação cruzada

**Configurar as 5 divisões**

In [8]:
cross_val = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

- `n_splits=5`: divide os dados em 5 conjuntos
- `shuffle=True`: embaralha os dados antes de dividi-los
- `random_state=42`: permite reproduzir as mesmas divisões

**Minha previsão**

Minha previsão é que a média da validação cruzada fique próxima dos 81% obtidos anteriormente no pipeline. Acredito que haverá alguma variação entre as rodadas, mas espero que o desempenho seja relativamente consistente. A validação cruzada permitirá verificar se a acurácia anterior foi representativa ou se pode ter sido influenciada pela divisão específica dos dados.

**Executando a validação cruzada**

In [9]:
cv_scores = cross_val_score(
    model_pipeline,             # pipeline completo, incluindo pre-processamento e regressão logística
    X_train,                    # variavel preditora do treino
    y_train,                    # alvo do treino
    cv=cross_val,               # configuração das 5 divisões
    scoring="accuracy"          # metrica utilizada para cada rodada
)

**Resultados da validação cruzada**

In [10]:
# resultados individuais
for i, score in enumerate(cv_scores, start=1):
    print(f"Rodada {i}: {score:.2%}")

# media e desvio padrão
mean_score = cv_scores.mean()
std_score = cv_scores.std()

print(f"\nMédia: {mean_score:.2%}")
print(f"Desvio padrão: {std_score:.2%}")

# faixa de média +- desvio padrão
lower_bound = mean_score - std_score
upper_bound = mean_score + std_score

print(f"Faixa: {lower_bound:.2%} a {upper_bound:.2%}")

Rodada 1: 79.73%
Rodada 2: 80.27%
Rodada 3: 79.59%
Rodada 4: 82.31%
Rodada 5: 81.63%

Média: 80.71%
Desvio padrão: 1.08%
Faixa: 79.63% a 81.79%


- **acurácia média**: desempenho médio nas cinco rodadas
- **desvio padrão**: indica o quanto os resultados variam em torno da média
- **faixa**: representa a média menos e mais um desvio padrão. É uma descrição da variação, não um intervalo de confiança formal.

As 5 rodadas apresentaram resultados bastante próximos, o que é um bom indicativo de consistência.

**Média:** o modelo apresentou aproximadamente 81% de acurácia, em média, nas cinco rodadas.

**Desvio padrão:** essa é a parte mais importante para entender estabilidade. O desvio padrão indica quanto os resultados variaram em relação à média. Nesse caso, foi de 1,08 ponto percentual. Isso é uma variação relativamente pequena: as rodadas tiveram desempenhos próximos de 80,71%.

**Faixa de desvio padrão:** a faixa de 79,63% a 81,79% foi calculada assim:
- Limite inferior: 80,71 − 1,08 = 79,63%.
- Limite superior: 80,71 + 1,08 = 81,79%.

---

**Sobre a expectativa de 87–94%:** minha expectativa inicial de 87–94% parece ter sido otimista para o modelo atual, considerando os resultados observados até aqui. Entretanto, ainda é necessário avaliar outras métricas, especialmente recall e F1-score, para compreender melhor o comportamento da classificação.


### Comparação com o resultado da fase 4

A acurácia obtida na fase 4 foi de **80,98%**, enquanto a validação cruzada apresentou uma média de **80,71%**, com faixa de um desvio padrão entre **79,63% e 81,79%**.

Como os 80,98% estão dentro dessa faixa, concluo que a divisão de treino e teste da fase 4 apresentou um resultado compatível com o desempenho observado nas cinco rodadas de validação cruzada. Isso sugere que o resultado anterior foi representativo do comportamento do modelo nessas divisões, e não apenas consequência de uma divisão particularmente favorável.

Minha previsão também se confirmou: esperava que a acurácia da fase 4 estivesse dentro da faixa de média mais ou menos um desvio padrão, e isso aconteceu.

A validação cruzada reforça a consistência do desempenho em torno de 81%, mas não garante a generalização para hospitais que não participaram do treinamento. Essa questão será investigada na fase 6.

## Matriz de confusão

O conjunto de teste tem 184 pacientes, e a acurácia anterior foi de 80,98%. Isso significa que foram 149 classificações corretas e 35 classificações incorretas. Mas a acurácia não indicou como os 35 erros estão divididos entre falsos positivos (FP) e falsos negativos (FN).

**Minha previsão**

Considerando os aproximadamente 35 erros da avaliação anterior, minha hipótese é que haverá tanto falsos positivos quanto falsos negativos. Espero uma distribuição relativamente próxima entre os dois tipos, mas ainda não tenho evidências para confirmar qual será a mais frequente. A matriz de confusão permitirá identificar essa distribuição.

**Treinar o modelo novamente**

In [11]:
# treinar o pipeline com os dados de treino
model_pipeline.fit(X_train, y_train)

# fazer previsões com os dados de teste
y_pred = model_pipeline.predict(X_test)

- `fit()`: treina o pipeline com os 736 pacientes de treino
- `predict()`: utiliza o modelo treinado para classificar os 184 pacientes de teste
- `y_pred`: armazena as classificações previstas pelo modelo 

**Calcular a matriz de confusão**

In [12]:
cm = confusion_matrix(
    y_test,
    y_pred,
    labels=[0, 1] # 0: paciente saudável, 1: paciente doente
)

print(cm)

[[63 19]
 [16 86]]


Como ler a saída do `print(cm)`:

| | Previsto: saudável (0) | Previsto: doente (1) |
| --- | :---: | :---: |
| **Real: saudável (0)** | ✅ VN | ❌ FP |
| **Real: doente (1)** | ❌ FN | ✅ VP |

| Resultado | Significado clínico |
| --- | --- |
| VN | Pacientes sem doença que foram corretamente classificados como saudáveis. |
| FP | Pacientes sem doença que foram classificados como doentes. São falsos alarmes. |
| FN | Pacientes com doença que foram classificados como saudáveis. São casos que o modelo não identificou. |
| VP | Pacientes com doença que foram corretamente classificados como doentes. |

**Falso negativo × falso positivo**

- **Falso negativo:** o modelo deixa passar um paciente doente. Isso pode gerar falsa tranquilidade e atrasar uma investigação clínica.

- **Falso positivo:** o modelo sinaliza doença em um paciente saudável. Pode gerar preocupação e levar a exames adicionais desnecessários.

**Deixando a matriz mais legível**

In [13]:
matrix_df = pd.DataFrame(
    cm,
    index=["Real: saudável (0)", "Real: doente (1)"],
    columns=["Previsto: saudável (0)", "Previsto: doente (1)"]
)

display(matrix_df)

,Previsto: saudável (0),Previsto: doente (1)
Real: saudável (0),63,19
Real: doente (1),16,86


**Interpretação da matriz de confusão**

- VN = 63 pacientes sem doença que foram corretamente classificados como saudáveis.
- FP = 19 pacientes sem doença que foram classificados como doentes.
- FN = 16 pacientes com doença que foram classificados como saudáveis.
- VP = 86 pacientes com doença que foram corretamente classificados como doentes.

Total de erros: FP + FN = 35 pacientes.

No total, o modelo realizou 149 classificações corretas e 35 incorretas, mantendo aproximadamente 81% de acurácia.

Minha previsão de que os falsos positivos e falsos negativos teriam quantidades relativamente próximas se confirmou. Foram 19 falsos positivos e 16 falsos negativos.

O recall da classe positiva foi de aproximadamente 84,31%, indicando que o modelo identificou 86 dos 102 pacientes que realmente apresentavam doença. Já a especificidade foi de aproximadamente 76,83%.

Apesar do resultado inicial, os 16 falsos negativos merecem atenção, pois representam pacientes doentes classificados como saudáveis. Portanto, a acurácia isoladamente não é suficiente para avaliar a qualidade do modelo em um problema de classificação de doença cardíaca.

**Qual erro é pior?**

Considero mais importante reduzir os falsos negativos, pois representam pacientes doentes classificados como saudáveis, podendo atrasar a investigação e o tratamento. Já os falsos positivos podem gerar preocupação e exames desnecessários. Por isso, priorizo a identificação correta dos pacientes doentes, mesmo que isso aumente os falsos positivos. Nesse contexto, o recall da classe positiva é uma métrica importante. O modelo deve ser utilizado apenas como ferramenta experimental de apoio, nunca como substituto do diagnóstico médico.